In [29]:
"""
Reciprocal Rank Fusion (RRF) RAG Pipeline -- Production-Grade
==============================================================
Architecture   : FIVE configurations covering the complete RRF design space
                 as established by the 2009-2025 research literature:
                 (1) Baseline                     -- single dense FAISS retriever,
                                                     no fusion
                 (2) Standard RRF (k=60)          -- BM25 + FAISS via EnsembleRetriever,
                                                     equal weights, c=60 (vanilla RRF)
                 (3) Weighted RRF + k-sweep       -- BM25 + FAISS + MMR, heterogeneous
                                                     weights, custom c values per query
                                                     class, manual RRF implementation
                                                     exposing full score transparency
                 (4) RAG-Fusion (Multi-Query RRF) -- LLM generates N query variants,
                                                     each variant retrieves independently,
                                                     all result lists fused by RRF
                                                     (Rackauckas, arXiv 2402.03367)
                 (5) Two-Stage RRF + CrossEncoder [BEST]
                                                  -- Stage 1: broad RRF candidate pool
                                                     (K=20 per retriever, 3 retrievers)
                                                     Stage 2: CrossEncoder precision
                                                     reranking on top-50 RRF candidates

Vector Store   : FAISS  (IndexFlatIP, BGE-large-en-v1.5, 1024-dim)
BM25           : BM25Plus (rank-bm25)
Embeddings     : BAAI/bge-large-en-v1.5  (1024-dim, asymmetric bi-encoder)
CrossEncoder   : BAAI/bge-reranker-large (~700MB, for Config 5)
LLM            : Azure OpenAI  (ChatOllama -- GPT-4o)
Framework      : LangChain 0.2+ (EnsembleRetriever, ContextualCompressionRetriever,
                                  CrossEncoderReranker)

Reference PDFs (open-access arXiv):
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf
    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf
    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
What Reciprocal Rank Fusion Is and Why It Works
=============================================================================
Reciprocal Rank Fusion (RRF) is a rank-based, non-metric fusion technique used
to combine multiple ranked lists -- typically from diverse retrieval models or
query formulations -- into a single consensus ranking.

Original paper: Cormack, Clarke, Buettcher (2009) SIGIR, "Reciprocal rank fusion
outperforms condorcet and individual rank learning methods."

The core formula for a document D given M retrieval systems:

    RRF_score(D) = SUM_{i=1}^{M}  [  w_i / (k + rank_i(D))  ]

Where:
    rank_i(D) : the 1-based position of document D in retrieval system i's ranked list.
                If D does not appear in system i's list: contributes 0 to the sum.
    k          : the smoothing constant (default 60 in original paper and LangChain).
                 Determines how sharply the formula penalizes lower-ranked documents.
    w_i        : the weight assigned to retrieval system i (default 1.0 for pure RRF;
                 LangChain's EnsembleRetriever supports non-uniform weights).
    M          : number of retrieval systems.

=============================================================================
The Mathematics of k: Why 60 Is the Default and When to Change It
=============================================================================
The k constant fundamentally determines the shape of the RRF score distribution:

AT k=1 (aggressive):
    rank 1 -> score = 1/(1+1)  = 0.500
    rank 5 -> score = 1/(1+5)  = 0.167
    Ratio (rank1/rank5) = 3.0x

AT k=60 (standard):
    rank 1 -> score = 1/(60+1) = 0.01639
    rank 5 -> score = 1/(60+5) = 0.01538
    Ratio (rank1/rank5) = 1.065x

AT k=1000 (flat):
    rank 1 -> score = 1/(1000+1) = 0.000999
    rank 5 -> score = 1/(1000+5) = 0.000995
    Ratio (rank1/rank5) = 1.004x

INTERPRETATION:
    Small k (1-10):   The top-ranked document in any single retriever gets a massive
                      score advantage. Even if only ONE retriever places D at rank 1,
                      D is likely to win the RRF ranking, regardless of how other
                      retrievers ranked it. High reward for consensus is minimal;
                      instead, high reward for local dominance.
                      Use when: you trust one retriever very highly (e.g., domain-tuned
                      dense retriever on a specialized corpus).

    k=60 (standard):  The score difference between rank 1 and rank 60 is approximately
                      2x (0.01639 vs 0.01613). A document at rank 60 in one retriever
                      still contributes meaningfully to its RRF total. This makes RRF
                      robust to the case where the correct document is ranked 20th by
                      retriever A but 5th by retriever B: its combined score will exceed
                      a document that is only 1st in retriever A but absent from B.
                      The "consensus effect" at k=60 is what makes RRF the industry
                      baseline for hybrid retrieval.
                      k=60 is the most common default value (Bruch et al., 2022,
                      Rackauckas, 2024).

    Large k (100+):   Very flat scoring; almost all documents receive nearly equal
                      scores. The consensus effect dominates: only documents appearing
                      in MULTIPLE lists will clearly outperform single-list documents.
                      Use when: the retrievers are high-quality and disagreement is
                      meaningful (if only one retriever finds a doc at rank 1, the doc
                      may genuinely not be relevant).

PRODUCTION CALIBRATION:
    k values tuned for one data distribution often generalize poorly to others.
    A coarse sweep (k in {5, 20, 60, 120, 300}) on your eval set is the minimum
    calibration requirement before production deployment.
    If no labeled data is available: default to k=60, it is the best zero-shot choice.

=============================================================================
Why RRF Does Not Need Score Normalization
=============================================================================
The central practical advantage of RRF over score-based fusion (Convex Combination, CC):
RRF merges results purely based on rank using the formula 1/(k + rank). It avoids
score calibration problems, rewards agreement across retrievers, and prevents dominance
of any single system using a large constant k (usually 60).

BM25 returns TF-IDF weighted term frequency scores. For a 100-document corpus:
    BM25 score range: approximately 0 to 25.
    Typical query: BM25 returns scores like [14.2, 11.8, 9.3, ...]

BGE-large cosine similarity returns normalized dot products. For the same query:
    Dense score range: 0.70 to 0.95 (after L2 normalization).
    Typical query: dense returns scores like [0.93, 0.88, 0.85, ...]

These two score scales are completely incomparable. Naive addition (14.2 + 0.93)
is meaningless because the BM25 score dominates by 15x. Min-max normalization to
[0,1] before addition requires knowing the corpus-level min and max BM25 score,
which varies with corpus size and vocabulary. RRF sidesteps ALL of this: it only
uses the rank position (1, 2, 3, ...) which is scale-invariant.

=============================================================================
The Consensus Effect: Why Documents in Multiple Lists Win
=============================================================================
The most powerful property of RRF for RAG systems: a document that appears in
MULTIPLE retrieval lists accumulates score from EACH list it appears in:

    Document A: rank 1 in BM25, NOT found by dense -> 0.01639 + 0 = 0.01639
    Document B: rank 5 in BM25, rank 3 in dense    -> 0.01538 + 0.01587 = 0.03125

Document B wins despite ranking 5th in BM25, because both retrievers agree it
is relevant. This is the consensus effect: RRF is biased toward documents that
are independently validated by multiple retrieval systems.

For RAG systems, this consensus is quality assurance: a document that BM25 finds
(exact keyword match) AND dense retrieval finds (semantic relevance) has been
validated from two orthogonal perspectives. Documents validated by both approaches
are significantly less likely to be false positives than documents validated by
only one approach.

=============================================================================
RAG-Fusion: Multi-Query RRF (Rackauckas, arXiv 2402.03367)
=============================================================================
RAG-Fusion combines RAG and RRF by generating multiple queries, reranking them
with reciprocal scores and fusing the documents and scores. RAG-Fusion was able
to provide accurate and comprehensive answers due to the generated queries
contextualizing the original query from various perspectives.

RAG-Fusion extends the consensus effect across QUERY SPACE rather than just
retrieval method space:
    Standard RRF: consensus across retrievers (BM25 vs dense) for ONE query.
    RAG-Fusion:   consensus across queries (original vs paraphrase vs sub-question)
                  for ONE retrieval method.
    Combined:     consensus across BOTH retriever diversity AND query diversity.

The production constraint: when number of queries m > 5, informational redundancy
outweighs recall gains. Effective usage prescribes 3-5 well-crafted sub-queries.
RAG-Fusion increases runtime by an average 1.77x due to parallel retrieval costs.
The topic drift risk: if a generated sub-query diverges from the original intent,
its retrieved documents introduce off-topic content that can confuse the LLM.
Mitigation: include the original query in the fusion (with highest weight) to
anchor the merged results to the user's actual intent.

=============================================================================
Production Finding: RRF Recall Gains vs. Post-Reranking Neutralization
=============================================================================
A 2025 production-scale RAG study (arXiv 2603.02153) found that retrieval fusion
does increase raw recall, but these gains are largely neutralized after re-ranking
and truncation to a fixed context window. In their setting, fusion variants fail to
outperform single-query baselines on KB-level Top-K metrics after reranking.

This finding is the design motivation for Config 5 (Two-Stage RRF + CrossEncoder):
    Stage 1: RRF with a WIDE candidate pool (K=20 per retriever).
             Purpose: maximize recall -- ensure the correct document IS in the pool.
    Stage 2: CrossEncoder precision reranking on the top-50 RRF candidates.
             Purpose: maximize precision -- ensure the correct document is in the
             top-4 positions that the LLM actually reads.

The recall-precision handoff (RRF wide pool -> CrossEncoder narrow selection)
addresses exactly the neutralization problem: RRF improves recall but not precision;
CrossEncoder improves precision but needs candidates to already be in the pool.
Together they achieve both.

"""

'\nReciprocal Rank Fusion (RRF) RAG Pipeline -- Production-Grade\n==============================================================\nArchitecture   : FIVE configurations covering the complete RRF design space\n                 as established by the 2009-2025 research literature:\n                 (1) Baseline                     -- single dense FAISS retriever,\n                                                     no fusion\n                 (2) Standard RRF (k=60)          -- BM25 + FAISS via EnsembleRetriever,\n                                                     equal weights, c=60 (vanilla RRF)\n                 (3) Weighted RRF + k-sweep       -- BM25 + FAISS + MMR, heterogeneous\n                                                     weights, custom c values per query\n                                                     class, manual RRF implementation\n                                                     exposing full score transparency\n                 (4) RAG-Fusion (Multi-Query 

In [30]:
import os
import sys
import time
import pickle
import logging
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any

import numpy as np


In [31]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser


In [32]:

# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("rrf_rag")


In [33]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class RRFConfig:
    """
    Centralized configuration for the RRF RAG pipeline.

    RRF_K (60):
        The smoothing constant in the RRF formula: score = w / (k + rank).
        k=60 is the universal default established in Cormack et al. 2009.
        At k=60, rank-1 document score = 0.01639, rank-60 score = 0.01587.
        The ~3% score difference at the tails means deep-ranked documents
        still contribute meaningfully to the total -- the consensus effect is strong.
        Production sweep recommendation: test k in {20, 60, 120} on your labeled
        eval set before final deployment.

    RRF_K_SEMANTIC (20):
        A lower k for purely semantic queries (no exact keyword match expected).
        With k=20: rank-1 = 0.0476, rank-5 = 0.0400, ratio = 1.19x.
        More discriminative at the top of the ranking -- rewards dense retriever's
        top-ranked documents more aggressively than k=60.

    RRF_K_KEYWORD (120):
        A higher k for exact-keyword queries (product codes, names, error codes).
        With k=120: rank-1 = 0.00826, rank-5 = 0.00800, ratio = 1.03x.
        Very flat scoring -- the consensus effect dominates. Prevents BM25 from
        dominating the fusion on keyword queries where it may rank docs very
        highly that are not actually the most relevant for the full question.

    RRF_WINDOW_K (20):
        The number of candidates each retriever returns before RRF fusion in
        two-stage setups. A wider window (20 vs 5) increases recall at the cost
        of passing more candidates to the downstream CrossEncoder.
        The rank fusion window M and reranker window N are separate parameters:
        each search mode produces M results, fusion combines to 2M results cut
        to M, then reranker window N is applied, then final top-K is returned.

    RAGFUSION_N_QUERIES (4):
        Number of LLM-generated query variants for RAG-Fusion (Config 4).
        Include the original query for a total of N+1 unique queries.
        At N=4 queries: total retrieval calls = 5 (4 generated + 1 original).
        When m > 5, informational redundancy outweighs recall gains.
        Production recommendation: N=3 for balance of quality and latency.

    CROSS_ENCODER_MODEL:
        BAAI/bge-reranker-large for Config 5.
        ~700 MB model. Processes (query, document) pairs jointly with full attention.
        10x slower than bi-encoder but 15-30% better precision at top-K selection.
        Batch size 16 is the production-calibrated value for CPU inference.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",    "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers", "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",  "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- FAISS -------------------------------------------------------------
    FAISS_PERSIST_DIR: str = "./faiss_rrf_index"

    # --- BM25 --------------------------------------------------------------
    BM25_PERSIST_DIR: str  = "./bm25_rrf_index"
    BM25_PARAMS:      Dict = {"k1": 1.5, "b": 0.75, "delta": 0.5}

    # --- BGE Embeddings ----------------------------------------------------
    BIENCODER_MODEL:             str = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE:            str = "cpu"
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- CrossEncoder (Config 5) -------------------------------------------
    CROSS_ENCODER_MODEL:      str = "BAAI/bge-reranker-large"
    CROSS_ENCODER_BATCH_SIZE: int = 16
    CROSS_ENCODER_TOP_N:      int = 4     # final docs after reranking

    # --- RRF k Parameter Variants ------------------------------------------
    RRF_K:          int = 60    # standard: balanced consensus/discrimination
    RRF_K_SEMANTIC: int = 20    # aggressive: rewards top-ranked docs more
    RRF_K_KEYWORD:  int = 120   # flat: maximizes consensus effect

    # --- Retrieval K Parameters --------------------------------------------
    BASE_K:       int = 5    # baseline per-retriever K
    RRF_WINDOW_K: int = 20   # wide candidate pool for two-stage RRF
    FINAL_K:      int = 4    # documents passed to LLM

    # --- EnsembleRetriever Weights ----------------------------------------
    # Weights in LangChain EnsembleRetriever are applied as per-retriever
    # multipliers on the RRF score: weighted_score = w * (1 / (c + rank))
    # Pure RRF uses equal weights (all 1.0, or normalized to sum=1).
    WEIGHTS_EQUAL:    List[float] = [0.5, 0.5]          # Config 2: BM25 + dense
    WEIGHTS_3WAY:     List[float] = [0.3, 0.5, 0.2]     # Config 3: BM25 + dense + MMR
    WEIGHTS_WIDE:     List[float] = [0.33, 0.67]         # Config 5: BM25 + dense

    # --- RAG-Fusion -------------------------------------------------------
    RAGFUSION_N_QUERIES:    int   = 4      # generated variants (+ original = 5 total)
    RAGFUSION_TEMPERATURE:  float = 0.3   # diversity in generated queries
    RAGFUSION_INCLUDE_ORIG: bool  = True  # always include original query in fusion

    # --- Text Splitting ---------------------------------------------------
    CHUNK_SIZE:    int = 450
    CHUNK_OVERLAP: int = 60

    # --- LLM Temperature --------------------------------------------------
    LLM_ANSWER_TEMPERATURE:    float = 0.0
    LLM_EXPANSION_TEMPERATURE: float = 0.3

    # --- Azure OpenAI LLM -------------------------------------------------
    OLLAMA_BASE_URL: str = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.environ.get("OLLAMA_MODEL",    "qwen2.5-coder:7b")
    LLM_MAX_TOKENS: int           = 1024

    # --- Prompts ----------------------------------------------------------
    RAG_PROMPT: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer is not present, respond:
"The provided documents do not contain sufficient information to answer this question."

Context (retrieved via RRF fusion):
{context}

Question: {question}

Precise technical answer:"""

    RAGFUSION_QUERY_GEN_PROMPT: str = """You are an expert at query reformulation for
information retrieval. Given the original user question, generate {n} distinct
retrieval queries that together cover different aspects, phrasings, and subtopics
of the original question. Each query should be:
- Self-contained and specific
- Phrased differently from the original and from each other
- Semantically related to the original (do NOT introduce unrelated topics)

Output ONLY the queries, one per line. No numbering, no explanation.

Original question: {question}

{n} reformulated queries:"""


In [34]:

# ===========================================================================
# SECTION 2 -- RRF TRACE DATA CLASS
# ===========================================================================

@dataclass
class RRFTrace:
    """
    Records the full execution trace of one RRF RAG run.

    rrf_scores: dict mapping doc_id -> final RRF score for the top-K results.
        Exposes the actual numeric RRF scores, enabling score distribution analysis.
        In production, monitoring the score distribution reveals:
          - High score concentration (top doc >> rest): query has a clear best match.
          - Flat score distribution (all docs similar scores): ambiguous query;
            consider query rewriting or clarification.

    retriever_hits: dict of {retriever_name: n_docs_returned}.
        Monitors per-retriever contribution. If BM25 consistently returns 0
        for a query class, that class is semantically-only and BM25 is dead weight.

    query_variants: populated by Config 4 (RAG-Fusion) with the LLM-generated
        query variants. Enables auditing topic drift in generated queries.
        If any variant has cosine similarity < 0.6 to the original query embedding,
        flag it as a potential drift candidate.

    k_used: the actual k parameter used for this RRF run. Enables auditing
        Config 3's adaptive k behavior: was the right k class selected?
    """
    question:         str
    strategy:         str
    final_docs:       List[Document] = field(default_factory=list)
    rrf_scores:       Dict[str, float] = field(default_factory=dict)
    retriever_hits:   Dict[str, int]   = field(default_factory=dict)
    query_variants:   List[str]        = field(default_factory=list)
    k_used:           int              = 60
    candidates_pre_rerank: int         = 0
    final_answer:     str              = ""
    retrieval_ms:     float            = 0.0
    rerank_ms:        float            = 0.0
    generation_ms:    float            = 0.0
    total_ms:         float            = 0.0

    def print_trace(self) -> None:
        print(f"\n{'='*74}")
        print(f"TRACE: {self.strategy}")
        print(f"Query: {self.question[:80]}")
        print(f"{'='*74}")
        print(f"\n  k constant used: {self.k_used}")
        print(f"  Candidates pre-rerank: {self.candidates_pre_rerank}")
        print(f"  Final docs to LLM:     {len(self.final_docs)}")

        if self.retriever_hits:
            print(f"\n  Retriever hits:")
            for name, hits in self.retriever_hits.items():
                print(f"    {name:<40}: {hits} docs")

        if self.query_variants:
            print(f"\n  Query variants (RAG-Fusion):")
            for i, q in enumerate(self.query_variants, 1):
                print(f"    [{i}] {q[:70]}")

        if self.rrf_scores:
            print(f"\n  RRF scores (top docs):")
            for doc_id, score in list(self.rrf_scores.items())[:5]:
                try:
                    score_display = f"{float(score):.6f}"
                except (TypeError, ValueError):
                    score_display = str(score)
                print(f"    {doc_id[:50]:<50}  {score_display}")

        print(f"\n  Final docs:")
        for i, doc in enumerate(self.final_docs, 1):
            paper = doc.metadata.get("paper_name", "?")[:30]
            page  = doc.metadata.get("page", "?")
            score = self.rrf_scores.get(f"doc_{i}", "n/a")
            print(f"    [{i}] {paper} p{page}  | rrf={score}")

        print(f"\n  Latency: retrieval={self.retrieval_ms:.0f}ms  "
              f"rerank={self.rerank_ms:.0f}ms  "
              f"gen={self.generation_ms:.0f}ms  "
              f"total={self.total_ms:.0f}ms")
        print(f"\n  ANSWER:\n  {self.final_answer[:450]}")
        print("=" * 74 + "\n")


In [35]:

# ===========================================================================
# SECTION 3 -- CORE RRF IMPLEMENTATION (Manual, Score-Transparent)
# ===========================================================================

def rrf_fuse(
    ranked_lists:     List[List[Document]],
    k:                int   = 60,
    weights:          Optional[List[float]] = None,
    dedup_key:        str   = "page_content",
) -> Tuple[List[Document], Dict[str, float]]:
    """
    Manual Reciprocal Rank Fusion implementation with full score transparency.

    Unlike EnsembleRetriever (which hides internal RRF scores), this function
    returns the exact RRF score for every document in the merged result, enabling
    production monitoring, debugging, and adaptive k selection.

    Formula implemented:
        RRF_score(D) = SUM_{i=1}^{M} [ w_i / (k + rank_i(D)) ]

    Where:
        rank_i(D)  = 1-based position in list i. If D not in list i: contributes 0.
        k          = smoothing constant (prevents rank-1 from dominating).
        w_i        = per-list weight (default 1.0 for all lists).
        M          = total number of ranked lists being fused.

    Deduplication strategy:
        Documents are deduplicated by their first DEDUP_PREFIX_LEN=150 chars
        of page_content. This handles the common case where the same chunk
        appears at different rank positions in two retrievers (e.g., BM25 rank 2,
        dense rank 1). Only the FIRST occurrence (in the fused order after RRF)
        is kept; subsequent occurrences are dropped but their scores are accumulated
        into the first occurrence's RRF total BEFORE deduplication.

    Args:
        ranked_lists : List of M ranked Document lists (each from one retriever).
        k            : RRF smoothing constant. Default 60.
        weights      : Per-list weight multipliers. Default: uniform 1.0 per list.
        dedup_key    : Deduplication field. "page_content" for content-based dedup.

    Returns:
        Tuple of:
            - List[Document]: documents sorted by descending RRF score (deduped).
            - Dict[str, float]: mapping content_prefix -> final RRF score.
    """
    if weights is None:
        weights = [1.0] * len(ranked_lists)
    assert len(weights) == len(ranked_lists), \
        f"weights length {len(weights)} != lists length {len(ranked_lists)}"

    # Accumulate RRF scores across all lists
    score_map:  Dict[str, float]    = {}   # content_prefix -> rrf_score
    doc_map:    Dict[str, Document] = {}   # content_prefix -> Document object

    for list_idx, (doc_list, w) in enumerate(zip(ranked_lists, weights)):
        for rank_0based, doc in enumerate(doc_list):
            rank_1based = rank_0based + 1
            prefix = doc.page_content[:150]   # deduplication key

            rrf_contribution = w / (k + rank_1based)
            score_map[prefix] = score_map.get(prefix, 0.0) + rrf_contribution

            if prefix not in doc_map:
                doc_map[prefix] = doc

    # Sort by descending RRF score
    sorted_prefixes = sorted(score_map, key=lambda p: score_map[p], reverse=True)
    fused_docs   = [doc_map[p] for p in sorted_prefixes]
    fused_scores = {p: score_map[p] for p in sorted_prefixes}

    return fused_docs, fused_scores



In [36]:

# ===========================================================================
# SECTION 4 -- PDF, INDEX, AND MODEL BUILDERS
# ===========================================================================

def download_pdfs(config: RRFConfig) -> List[Path]:
    """Download research PDFs with file-existence caching."""
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []
    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            logger.info("Cached: %s  (%.1f KB)", dest.name, dest.stat().st_size / 1024)
            paths.append(dest)
            continue
        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (RAG-RRF-Pipeline/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            if len(data) < 10_000:
                raise RuntimeError(f"File too small: {url}")
            dest.write_bytes(data)
            logger.info("Saved: %s  (%.1f KB)", dest.name, len(data) / 1024)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(
                f"Cannot download '{url}'. Place manually at '{dest}'."
            ) from exc
    return paths


def load_and_chunk_documents(
    pdf_paths: List[Path],
    config:    RRFConfig,
) -> List[Document]:
    """Load PDFs and split into chunks with paper_name and page metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        add_start_index=True,
    )
    all_chunks: List[Document] = []
    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        try:
            pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc
        for page in pages:
            page.metadata.update({"source": pdf_path.name, "paper_name": paper_name})
        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d pages -> %d chunks", pdf_path.name, len(pages), len(chunks))
        all_chunks.extend(chunks)
    logger.info("Total chunks: %d", len(all_chunks))
    return all_chunks


def build_bge_embeddings(config: RRFConfig) -> HuggingFaceEmbeddings:
    logger.info("Loading BGE: %s", config.BIENCODER_MODEL)
    return HuggingFaceEmbeddings(
        model_name=config.BIENCODER_MODEL,
        model_kwargs={"device": config.BIENCODER_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_faiss_index(
    chunks:     List[Document],
    embeddings: HuggingFaceEmbeddings,
    config:     RRFConfig,
) -> FAISS:
    faiss_file = Path(config.FAISS_PERSIST_DIR) / "index.faiss"
    if faiss_file.exists():
        logger.info("Loading FAISS from '%s' ...", config.FAISS_PERSIST_DIR)
        try:
            vs = FAISS.load_local(
                config.FAISS_PERSIST_DIR, embeddings,
                allow_dangerous_deserialization=True,
            )
            logger.info("  Loaded. Vectors: %d", vs.index.ntotal)
            return vs
        except Exception as exc:
            logger.warning("FAISS load failed (%s), rebuilding ...", exc)
    logger.info("Building FAISS from %d chunks ...", len(chunks))
    t0 = time.perf_counter()
    vs = FAISS.from_documents(chunks, embeddings)
    logger.info("  Built. Vectors: %d  (%.2fs)", vs.index.ntotal, time.perf_counter() - t0)
    Path(config.FAISS_PERSIST_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(config.FAISS_PERSIST_DIR)
    return vs


def bm25_preprocess_text(text: str) -> List[str]:
    """Pickle-safe BM25 tokenization function."""
    return text.lower().split()


def build_bm25_index(chunks: List[Document], config: RRFConfig) -> BM25Retriever:
    cache = Path(config.BM25_PERSIST_DIR) / "bm25plus.pkl"
    if cache.exists():
        logger.info("Loading BM25 from '%s' ...", cache)
        try:
            with open(cache, "rb") as f:
                state = pickle.load(f)
            return BM25Retriever(
                vectorizer=state["vectorizer"], docs=state["docs"],
                k=state.get("k", config.BASE_K),
                preprocess_func=state.get("preprocess_func", bm25_preprocess_text),
            )
        except Exception as exc:
            logger.warning("BM25 cache load failed (%s); rebuilding ...", exc)
            try:
                cache.unlink()
            except OSError:
                pass

    logger.info("Building BM25Plus from %d chunks ...", len(chunks))
    bm25_params = dict(config.BM25_PARAMS)

    try:
        ret = BM25Retriever.from_documents(
            chunks, bm25_variant="plus",
            bm25_params=bm25_params,
            preprocess_func=bm25_preprocess_text,
        )
    except TypeError as exc:
        # Compatibility: some langchain-community versions instantiate BM25Okapi
        # regardless of bm25_variant and reject BM25Plus-only params (e.g., delta).
        if "delta" not in str(exc):
            raise
        logger.warning(
            "BM25Plus parameters unsupported by this BM25Retriever version; falling back to BM25Okapi-safe params."
        )
        safe_params = {k: v for k, v in bm25_params.items() if k in {"k1", "b", "epsilon"}}
        ret = BM25Retriever.from_documents(
            chunks,
            bm25_params=safe_params,
            preprocess_func=bm25_preprocess_text,
        )

    ret.k = config.BASE_K
    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache, "wb") as f:
        pickle.dump({"vectorizer": ret.vectorizer, "docs": ret.docs, "k": ret.k},
                    f, protocol=pickle.HIGHEST_PROTOCOL)
    return ret


In [37]:


def build_ollama_llm(
    config:      RRFConfig,
    temperature: float = None,
) -> ChatOllama:
    """Connect to local Ollama and return a ChatOllama instance."""
    import urllib.request, urllib.error
    base_url = getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434')
    model    = getattr(config, 'OLLAMA_MODEL',    'qwen2.5-coder:7b')
    try:
        urllib.request.urlopen(base_url, timeout=3)
    except Exception as exc:
        raise RuntimeError(
            f"Ollama not reachable at {base_url}. Start Ollama and run: "
            f"ollama pull qwen2.5-coder:7b"
        ) from exc
    return ChatOllama(
        base_url=base_url,
        model=model,
        temperature=getattr(config, 'LLM_TEMPERATURE', temperature),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )
    return ChatOllama(
        base_url=getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434'),
        model=getattr(config, 'OLLAMA_MODEL', 'qwen2.5-coder:7b'),
        temperature=getattr(config, 'LLM_TEMPERATURE', 0.0),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )


In [38]:

def format_and_answer(
    question: str,
    docs:     List[Document],
    llm:      ChatOllama,
    config:   RRFConfig,
) -> Tuple[str, float]:
    context = "\n\n---\n\n".join([
        f"[{d.metadata.get('paper_name','?')[:30]} | p{d.metadata.get('page','?')}]\n"
        f"{d.page_content.strip()}"
        for d in docs
    ])
    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT)
    t0 = time.perf_counter()
    answer = (prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    return answer, (time.perf_counter() - t0) * 1000


def make_bm25(bm25_base: BM25Retriever, k: int) -> BM25Retriever:
    """Create a fresh BM25Retriever view with the given k, reusing the index."""
    return BM25Retriever(
        vectorizer=bm25_base.vectorizer, docs=bm25_base.docs,
        k=k, preprocess_func=bm25_base.preprocess_func,
    )


In [39]:

# ===========================================================================
# SECTION 5 -- FIVE RRF CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: Baseline -- single dense FAISS retriever
# ---------------------------------------------------------------------------

def run_config1_baseline(
    question:    str,
    vectorstore: FAISS,
    llm:         ChatOllama,
    config:      RRFConfig,
) -> RRFTrace:
    """
    Configuration 1: Single dense FAISS retriever (no RRF).

    Top-FINAL_K documents by cosine similarity. Control condition.
    No fusion, no reranking. Establishes recall and quality floor.
    """
    trace = RRFTrace(question=question, strategy="Config1_Baseline_SingleRetriever")
    t0    = time.perf_counter()

    ret  = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.FINAL_K},
    )
    t_ret = time.perf_counter()
    docs  = ret.invoke(question)
    trace.retrieval_ms   = (time.perf_counter() - t_ret) * 1000
    trace.final_docs     = docs[: config.FINAL_K]
    trace.retriever_hits = {"FAISS_BGE": len(docs)}
    trace.k_used         = 0   # no RRF in baseline

    answer, trace.generation_ms = format_and_answer(question, trace.final_docs, llm, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [40]:

# ---------------------------------------------------------------------------
# CONFIG 2: Standard RRF k=60 via EnsembleRetriever
# ---------------------------------------------------------------------------

def run_config2_standard_rrf(
    question:    str,
    vectorstore: FAISS,
    bm25_base:   BM25Retriever,
    llm:         ChatOllama,
    config:      RRFConfig,
) -> RRFTrace:
    """
    Configuration 2: Standard RRF (k=60) via LangChain EnsembleRetriever.

    Uses LangChain's built-in EnsembleRetriever which implements a weighted
    variant of RRF: score = w * (1 / (c + rank)) summed across retrievers.

    Two retrievers:
        BM25Plus (sparse, lexical)    weight=0.5
        FAISS+BGE (dense, semantic)   weight=0.5

    c=60: the LangChain EnsembleRetriever's c parameter corresponds directly
    to the k constant in the original RRF formula. Default value 60.

    IMPORTANT IMPLEMENTATION NOTE on LangChain's Weighted RRF:
        LangChain's EnsembleRetriever implements: score = w * (1 / (c + rank)).
        Pure RRF (Cormack 2009) uses: score = 1 / (k + rank) without per-list weights.
        Pure RRF does not need weights because it operates on ranks, which are
        already normalized (ranks range from 1 to K regardless of absolute scores).
        LangChain's weighted variant is a blend of RRF and Convex Combination (CC):
        it applies RRF's rank-transformation but CC's weighted summation.
        Evidence on the effectiveness of this weighted RRF variant is scarce.
        For pure RRF behavior: set weights=[1.0, 1.0] (equal, unnormalized).
        For prioritizing one retriever: weights=[0.3, 0.7] or similar.

    Args:
        question    : User query.
        vectorstore : FAISS + BGE-large index.
        bm25_base   : BM25Plus retriever.
        llm         : ChatOllama.
        config      : RRFConfig.

    Returns:
        RRFTrace with EnsembleRetriever-fused documents. Note: LangChain's
        EnsembleRetriever does NOT expose per-document RRF scores, so rrf_scores
        dict will be empty in this config. Use Config 3 for score transparency.
    """
    trace = RRFTrace(
        question=question, strategy="Config2_Standard_RRF_k60_EnsembleRetriever"
    )
    t0 = time.perf_counter()

    bm25_r  = make_bm25(bm25_base, config.BASE_K)
    dense_r = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.BASE_K},
    )

    # EnsembleRetriever with standard k=60 and equal weights
    # c=60 in LangChain corresponds to k=60 in the RRF formula
    ensemble = EnsembleRetriever(
        retrievers=[bm25_r, dense_r],
        weights=config.WEIGHTS_EQUAL,   # [0.5, 0.5]
        c=config.RRF_K,                 # 60
    )

    t_ret = time.perf_counter()
    docs  = ensemble.invoke(question)
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000

    trace.final_docs          = docs[: config.FINAL_K]
    trace.candidates_pre_rerank = len(docs)
    trace.k_used              = config.RRF_K
    trace.retriever_hits      = {
        "BM25Plus":    len(bm25_r.invoke(question)),
        "FAISS_BGE":   len(dense_r.invoke(question)),
    }

    logger.info(
        "Config2 Standard RRF (k=%d): %d total candidates -> top-%d",
        config.RRF_K, len(docs), len(trace.final_docs),
    )

    answer, trace.generation_ms = format_and_answer(question, trace.final_docs, llm, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [41]:

# ---------------------------------------------------------------------------
# CONFIG 3: Weighted RRF with Adaptive k + Manual Score Transparency
# ---------------------------------------------------------------------------

def classify_query_type(question: str) -> str:
    """
    Heuristic query type classifier for adaptive k selection.

    Query classification is intentionally lightweight (keyword-based) to avoid
    adding LLM latency to the retrieval path. For production, replace with a
    fine-tuned intent classifier (fastText or distilBERT) for higher accuracy.

    Returns: "keyword" | "semantic" | "hybrid"
    """
    question_lower = question.lower()

    # Indicators of exact-keyword queries
    keyword_signals = [
        any(c.isdigit() for c in question),  # numbers (scores, versions, dates)
        len(question.split()) <= 6,           # short = likely keyword
        "what is" in question_lower and len(question.split()) <= 8,
        "define" in question_lower,
        "abbreviation" in question_lower,
        "acronym" in question_lower,
    ]
    # Indicators of semantic queries
    semantic_signals = [
        "how does" in question_lower,
        "explain" in question_lower,
        "why" in question_lower,
        "compare" in question_lower,
        "relationship" in question_lower,
        "difference" in question_lower,
        len(question.split()) >= 15,   # long = likely conceptual/semantic
    ]

    kw_score  = sum(keyword_signals)
    sem_score = sum(semantic_signals)

    if kw_score > sem_score:
        return "keyword"
    elif sem_score > kw_score:
        return "semantic"
    else:
        return "hybrid"


def run_config3_weighted_rrf_adaptive_k(
    question:    str,
    vectorstore: FAISS,
    bm25_base:   BM25Retriever,
    llm:         ChatOllama,
    config:      RRFConfig,
) -> RRFTrace:
    """
    Configuration 3: Weighted RRF with Adaptive k + Manual Score Transparency.

    Extends Config 2 with two innovations:
        (a) Adaptive k selection based on query type classification.
        (b) Manual RRF implementation using rrf_fuse(), which exposes the
            exact per-document RRF scores for monitoring and debugging.

    Three retrievers fused with heterogeneous weights:
        BM25Plus (sparse)            weight=0.3   (good for keywords)
        FAISS similarity (dense)     weight=0.5   (primary semantic retriever)
        FAISS MMR (diverse dense)    weight=0.2   (captures diverse relevant docs)

    Adaptive k parameter selection:
        "keyword" queries  -> k=20  (rewards top-ranked docs more aggressively)
        "semantic" queries -> k=60  (standard balanced consensus)
        "hybrid" queries   -> k=120 (maximum consensus effect; both retrievers must agree)

    This reflects the research finding that k values tuned for one data
    distribution often generalize poorly to others. A production system with
    diverse query types benefits from query-adaptive k selection rather than
    a single fixed k for all queries.

    Args:
        question    : User query.
        vectorstore : FAISS + BGE-large index.
        bm25_base   : BM25Plus retriever.
        llm         : ChatOllama.
        config      : RRFConfig.

    Returns:
        RRFTrace with full rrf_scores dict populated by manual rrf_fuse().
    """
    trace = RRFTrace(question=question, strategy="Config3_WeightedRRF_AdaptiveK")
    t0    = time.perf_counter()

    # Classify query to select appropriate k
    query_type = classify_query_type(question)
    k_adaptive = {
        "keyword":  config.RRF_K_SEMANTIC,  # k=20, more discriminative
        "semantic": config.RRF_K,           # k=60, standard
        "hybrid":   config.RRF_K_KEYWORD,   # k=120, maximum consensus
    }.get(query_type, config.RRF_K)

    logger.info("Config3 AdaptiveRRF: query_type='%s' -> k=%d", query_type, k_adaptive)
    trace.k_used = k_adaptive

    # Build three retrievers
    bm25_r  = make_bm25(bm25_base, config.BASE_K)
    sim_r   = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.BASE_K},
    )
    mmr_r   = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": config.BASE_K, "fetch_k": config.BASE_K * 3, "lambda_mult": 0.7},
    )

    # Retrieve from each independently
    t_ret = time.perf_counter()
    bm25_docs = bm25_r.invoke(question)
    sim_docs  = sim_r.invoke(question)
    mmr_docs  = mmr_r.invoke(question)
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000

    trace.retriever_hits = {
        "BM25Plus":        len(bm25_docs),
        "FAISS_similarity": len(sim_docs),
        "FAISS_MMR":       len(mmr_docs),
    }

    # Manual RRF fusion with adaptive k and heterogeneous weights
    fused_docs, fused_scores = rrf_fuse(
        ranked_lists=[bm25_docs, sim_docs, mmr_docs],
        k=k_adaptive,
        weights=config.WEIGHTS_3WAY,   # [0.3, 0.5, 0.2]
    )

    trace.final_docs          = fused_docs[: config.FINAL_K]
    trace.candidates_pre_rerank = len(fused_docs)

    # Build labeled score dict for top-K final docs
    trace.rrf_scores = {}
    for i, doc in enumerate(trace.final_docs):
        prefix = doc.page_content[:150]
        label  = f"doc_{i+1}_{doc.metadata.get('paper_name','?')[:15]}"
        trace.rrf_scores[label] = fused_scores.get(prefix, 0.0)

    logger.info(
        "Config3 Weighted RRF (k=%d, type=%s): %d candidates -> top-%d",
        k_adaptive, query_type, len(fused_docs), len(trace.final_docs),
    )

    answer, trace.generation_ms = format_and_answer(question, trace.final_docs, llm, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [42]:

# ---------------------------------------------------------------------------
# CONFIG 4: RAG-Fusion (Multi-Query RRF)
# ---------------------------------------------------------------------------

def generate_query_variants(
    question:   str,
    n:          int,
    llm:        ChatOllama,
    config:     RRFConfig,
) -> List[str]:
    """
    Generate N query variants for RAG-Fusion using the LLM.

    Each variant should approach the original query from a different angle:
    different phrasing, different sub-questions, different vocabulary.
    The generated variants are included alongside the original query in
    the RRF fusion pool.

    Returns: list of N generated variants (original NOT included here;
    it is added by the caller if RAGFUSION_INCLUDE_ORIG=True).
    """
    prompt  = ChatPromptTemplate.from_template(config.RAGFUSION_QUERY_GEN_PROMPT)
    llm_exp = build_ollama_llm(config, temperature=config.RAGFUSION_TEMPERATURE)
    raw     = (prompt | llm_exp | StrOutputParser()).invoke(
        {"question": question, "n": n}
    )
    variants = [line.strip() for line in raw.strip().split("\n") if line.strip()]
    return variants[:n]


def run_config4_ragfusion(
    question:    str,
    vectorstore: FAISS,
    bm25_base:   BM25Retriever,
    llm:         ChatOllama,
    config:      RRFConfig,
) -> RRFTrace:
    """
    Configuration 4: RAG-Fusion -- Multi-Query RRF (Rackauckas, arXiv 2402.03367).

    RAG-Fusion combines RAG and RRF by generating multiple queries,
    reranking them with reciprocal scores, and fusing the documents and scores.

    Full pipeline:
        Step 1: LLM generates RAGFUSION_N_QUERIES=4 query variants.
        Step 2: Retrieve for the original query + 4 variants = 5 total queries.
                Each query retrieves from both BM25 and FAISS (10 doc lists total).
        Step 3: RRF fusion across all 10 lists simultaneously.
                Documents appearing in multiple lists (across both queries and
                retrieval methods) accumulate the highest RRF scores.
        Step 4: Top-FINAL_K documents by RRF score to LLM.

    Why the consensus effect is amplified in RAG-Fusion:
        A document that is relevant to the ORIGINAL query AND to the PARAPHRASE
        of that query AND to the SUB-QUESTION of that query is almost certainly
        genuinely relevant -- it has been validated by multiple independent
        formulations of the same information need.
        A document that only appears for one of the 5 query variants is likely
        a partial match or a lucky lexical hit, not a truly relevant document.

    Topic drift mitigation:
        Including the original query with equal weight as the generated variants
        anchors the RRF fusion to the user's actual intent. Generated queries
        that drift from the original will retrieve documents that only appear
        in their list (low consensus) and will not strongly influence the final ranking.

    LLM call budget: 1 (query generation) + 1 (answer generation) = 2 calls.
    Retrieval calls: (N+1) * 2 = 5 * 2 = 10 individual retrieval calls.
    Expected latency increase vs baseline: 1.77x (Rackauckas, 2024).

    Args:
        question    : Original user query.
        vectorstore : FAISS + BGE-large index.
        bm25_base   : BM25Plus retriever.
        llm         : ChatOllama for answer generation.
        config      : RRFConfig.

    Returns:
        RRFTrace with query_variants populated and full rrf_scores from multi-query fusion.
    """
    trace = RRFTrace(question=question, strategy="Config4_RAGFusion_MultiQueryRRF")
    t0    = time.perf_counter()

    # --- Step 1: Generate query variants --------------------------------
    logger.info("Config4 RAG-Fusion: generating %d query variants ...", config.RAGFUSION_N_QUERIES)
    variants = generate_query_variants(question, config.RAGFUSION_N_QUERIES, llm, config)

    # Always include the original query as the anchor (highest-trust query)
    all_queries = []
    if config.RAGFUSION_INCLUDE_ORIG:
        all_queries.append(question)
    all_queries.extend(variants)
    trace.query_variants = variants

    logger.info("Config4 RAG-Fusion: total queries = %d", len(all_queries))

    # --- Step 2: Retrieve for each query from both BM25 and FAISS ------
    t_ret = time.perf_counter()
    all_ranked_lists: List[List[Document]] = []
    bm25_r = make_bm25(bm25_base, config.BASE_K)
    dense_r = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.BASE_K},
    )

    for q in all_queries:
        bm25_result  = bm25_r.invoke(q)
        dense_result = dense_r.invoke(q)
        all_ranked_lists.append(bm25_result)
        all_ranked_lists.append(dense_result)

    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000

    trace.retriever_hits = {
        f"BM25_q{i}":  len(all_ranked_lists[2*i])
        for i in range(len(all_queries))
    }
    trace.retriever_hits.update({
        f"FAISS_q{i}": len(all_ranked_lists[2*i+1])
        for i in range(len(all_queries))
    })

    # --- Step 3: RRF fusion across all lists ---------------------------
    # All lists use equal weight 1.0 (pure RRF -- no retriever gets extra credit)
    # The consensus effect naturally amplifies documents that appear in many lists
    n_lists = len(all_ranked_lists)
    equal_weights = [1.0] * n_lists

    fused_docs, fused_scores = rrf_fuse(
        ranked_lists=all_ranked_lists,
        k=config.RRF_K,
        weights=equal_weights,
    )

    trace.k_used              = config.RRF_K
    trace.candidates_pre_rerank = len(fused_docs)
    trace.final_docs          = fused_docs[: config.FINAL_K]

    # Populate score dict for final docs
    for i, doc in enumerate(trace.final_docs):
        prefix = doc.page_content[:150]
        label  = f"doc_{i+1}_{doc.metadata.get('paper_name','?')[:15]}"
        trace.rrf_scores[label] = fused_scores.get(prefix, 0.0)

    logger.info(
        "Config4 RAG-Fusion: %d queries x 2 retrievers = %d lists, "
        "%d unique candidates -> top-%d",
        len(all_queries), n_lists, len(fused_docs), len(trace.final_docs),
    )

    answer, trace.generation_ms = format_and_answer(question, trace.final_docs, llm, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace


In [43]:

# ---------------------------------------------------------------------------
# CONFIG 5: Two-Stage RRF + CrossEncoder [BEST]
# ---------------------------------------------------------------------------

def run_config5_twostage_rrf_crossencoder(
    question:     str,
    vectorstore:  FAISS,
    bm25_base:    BM25Retriever,
    llm:          ChatOllama,
    config:       RRFConfig,
) -> RRFTrace:
    """
    Configuration 5: Two-Stage RRF + CrossEncoder Reranker [BEST].

    Addresses the production finding (arXiv 2603.02153): RRF increases raw recall
    but gains are neutralized after reranking and truncation. The solution is to
    explicitly decouple the recall and precision optimization stages:

        STAGE 1 -- BROAD RRF CANDIDATE POOL (maximize recall):
            BM25Plus + FAISS similarity, both with K=RRF_WINDOW_K=20.
            EnsembleRetriever (c=60, weights=[0.33, 0.67]).
            Total candidates: up to 40 unique documents.
            Goal: ensure the ground-truth relevant documents are IN the pool.
            RRF does this well -- consensus effect surfaces genuinely relevant docs.

        STAGE 2 -- CROSSENCODER PRECISION RERANKING (maximize precision):
            CrossEncoderReranker(BAAI/bge-reranker-large, top_n=FINAL_K=4).
            Input: top-min(40, 50) documents from Stage 1.
            Cross-encoder processes each (query, document) pair jointly with full
            self-attention, providing a much more accurate relevance score than
            any bi-encoder or BM25 score.
            Goal: of the 40 candidates, identify the exact 4 that best answer the query.

    Why this two-stage design works where RRF alone does not:
        Stage 1 (RRF): O(1) per retriever. Scales to millions of documents.
                       Recall-oriented. Inexpensive.
        Stage 2 (CrossEncoder): O(n_candidates) pairs evaluated jointly.
                       Precision-oriented. Expensive but applied only to 40 docs.
        The cross-encoder's joint attention (query + document in the same forward
        pass) enables it to detect subtle semantic entailment, negation, and
        specificity that a bi-encoder's cosine similarity score cannot capture.

    LLM calls: 0 (answer generation only).
    CrossEncoder inference: min(2*K_window, 50) = min(40, 50) = 40 (query,doc) pairs.
    At 40 pairs, batch_size=16: 3 batches (~300ms CPU). GPU: ~30ms.

    Args:
        question    : User query.
        vectorstore : FAISS + BGE-large index.
        bm25_base   : BM25Plus retriever.
        llm         : ChatOllama.
        config      : RRFConfig.

    Returns:
        RRFTrace with rerank_ms populated and CrossEncoder-refined final docs.
    """
    trace = RRFTrace(
        question=question, strategy="Config5_TwoStage_RRF_CrossEncoder [BEST]"
    )
    t0 = time.perf_counter()

    # --- Stage 1: Wide RRF candidate pool (recall) ----------------------
    bm25_r  = make_bm25(bm25_base, config.RRF_WINDOW_K)
    dense_r = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.RRF_WINDOW_K},
    )

    # EnsembleRetriever: standard RRF k=60, sparse-biased weights
    # WEIGHTS_WIDE=[0.33, 0.67]: dense retriever gets higher weight because
    # BGE-large has stronger recall on semantic queries than BM25Plus
    ensemble = EnsembleRetriever(
        retrievers=[bm25_r, dense_r],
        weights=config.WEIGHTS_WIDE,   # [0.33, 0.67]
        c=config.RRF_K,                # 60
    )

    t_ret = time.perf_counter()
    stage1_docs = ensemble.invoke(question)
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000
    trace.k_used       = config.RRF_K

    trace.retriever_hits = {
        "BM25Plus":   len(bm25_r.invoke(question)),
        "FAISS_BGE":  len(dense_r.invoke(question)),
        "RRF_merged": len(stage1_docs),
    }
    trace.candidates_pre_rerank = len(stage1_docs)
    logger.info("Config5 Stage1 RRF: %d candidates", len(stage1_docs))

    # --- Stage 2: CrossEncoder precision reranking ----------------------
    t_rerank = time.perf_counter()
    try:
        cross_encoder = HuggingFaceCrossEncoder(
            model_name=config.CROSS_ENCODER_MODEL,
            model_kwargs={"device": "cpu"},
        )
        reranker = CrossEncoderReranker(
            model=cross_encoder,
            top_n=config.CROSS_ENCODER_TOP_N,
        )
        cc_retriever = ContextualCompressionRetriever(
            base_compressor=reranker,
            base_retriever=ensemble,
        )

        final_docs = cc_retriever.invoke(question)
        trace.rerank_ms  = (time.perf_counter() - t_rerank) * 1000
        trace.final_docs = final_docs[: config.FINAL_K]

    except Exception as exc:
        logger.warning(
            "Config5: CrossEncoder failed (%s). Falling back to RRF top-%d.",
            exc, config.FINAL_K,
        )
        trace.rerank_ms  = (time.perf_counter() - t_rerank) * 1000
        trace.final_docs = stage1_docs[: config.FINAL_K]

    # Assign pseudo-scores for trace (CrossEncoder scores stored in metadata by reranker)
    for i, doc in enumerate(trace.final_docs):
        ce_score = doc.metadata.get("relevance_score", f"rank_{i+1}")
        label    = f"doc_{i+1}_{doc.metadata.get('paper_name','?')[:15]}"
        trace.rrf_scores[label] = ce_score

    logger.info(
        "Config5 Two-Stage RRF+CE: %d stage1 -> %d final (CE reranked)",
        trace.candidates_pre_rerank, len(trace.final_docs),
    )

    answer, trace.generation_ms = format_and_answer(question, trace.final_docs, llm, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [44]:

# ===========================================================================
# SECTION 6 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question:    str,
    vectorstore: FAISS,
    bm25_base:   BM25Retriever,
    llm:         ChatOllama,
    config:      RRFConfig,
) -> Dict[str, RRFTrace]:
    """Run all five RRF configurations on a single question and print summary."""
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    runners = {
        "Config1_Baseline": lambda q: run_config1_baseline(
            q, vectorstore, llm, config),
        "Config2_StandardRRF_k60": lambda q: run_config2_standard_rrf(
            q, vectorstore, bm25_base, llm, config),
        "Config3_WeightedRRF_AdaptiveK": lambda q: run_config3_weighted_rrf_adaptive_k(
            q, vectorstore, bm25_base, llm, config),
        "Config4_RAGFusion_MultiQueryRRF": lambda q: run_config4_ragfusion(
            q, vectorstore, bm25_base, llm, config),
        "Config5_TwoStage_RRF_CrossEncoder [BEST]": lambda q: run_config5_twostage_rrf_crossencoder(
            q, vectorstore, bm25_base, llm, config),
    }

    traces: Dict[str, RRFTrace] = {}
    for label, fn in runners.items():
        print(f"\n{'='*62}\nRUNNING: {label}\n{'='*62}")
        try:
            trace = fn(question)
            trace.print_trace()
            traces[label] = trace
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    print("\n" + "=" * 78)
    print("RRF PIPELINE COMPARISON SUMMARY")
    print(f"{'Config':<48} {'k':>4} {'Cands':>6} {'Final':>6} {'Total(ms)':>10}")
    print("-" * 78)
    for lbl, tr in traces.items():
        print(
            f"{lbl:<48} {tr.k_used:>4} "
            f"{tr.candidates_pre_rerank:>6} "
            f"{len(tr.final_docs):>6} "
            f"{tr.total_ms:>10.0f}"
        )
    print("=" * 78 + "\n")

    return traces



In [45]:

# ===========================================================================
# SECTION 7 -- K SWEEP UTILITY (Production Calibration)
# ===========================================================================

def k_sweep_analysis(
    question:    str,
    vectorstore: FAISS,
    bm25_base:   BM25Retriever,
    k_values:    List[int] = None,
    config:      RRFConfig = None,
) -> None:
    """
    Production calibration utility: run RRF fusion at multiple k values
    and print the resulting document rankings side-by-side.

    This is the recommended first step before deploying any RRF RAG pipeline:
    run the k sweep on 10-20 representative labeled queries from your target
    domain and measure nDCG@K to select the optimal k value.

    At each k value, show:
        - The top-4 document sources and their RRF scores
        - How the ranking changes as k varies

    This directly illustrates the k tradeoff:
        k=5:   Highly discriminative. One retriever's rank-1 dominates.
        k=60:  Balanced. Consensus + some top-rank reward.
        k=300: Very flat. Only multi-list consensus documents stand out.
    """
    if k_values is None:
        k_values = [5, 20, 60, 120, 300]
    if config is None:
        config = RRFConfig()

    bm25_r  = make_bm25(bm25_base, config.BASE_K)
    dense_r = vectorstore.as_retriever(
        search_type="similarity", search_kwargs={"k": config.BASE_K},
    )
    bm25_docs  = bm25_r.invoke(question)
    dense_docs = dense_r.invoke(question)

    print(f"\n{'='*78}")
    print(f"RRF k SWEEP ANALYSIS")
    print(f"Query: {question[:70]}")
    print(f"BM25 top-1: {bm25_docs[0].page_content[:60] if bm25_docs else 'none'}...")
    print(f"Dense top-1: {dense_docs[0].page_content[:60] if dense_docs else 'none'}...")
    print(f"{'='*78}")
    print(f"{'k':>6}  {'Rank1 doc (paper p#, rrf_score)':50}")
    print("-" * 78)

    for k in k_values:
        fused, scores = rrf_fuse(
            ranked_lists=[bm25_docs, dense_docs],
            k=k, weights=[1.0, 1.0],
        )
        if fused:
            doc   = fused[0]
            paper = doc.metadata.get("paper_name", "?")[:20]
            page  = doc.metadata.get("page", "?")
            score = list(scores.values())[0]
            print(f"  k={k:>4}:  [{paper} p{page}]  score={score:.6f}")
    print("=" * 78 + "\n")


In [46]:
"""
    End-to-end RRF RAG pipeline orchestrator.

    LLM call counts per query:
        Config 1: 1 (answer)
        Config 2: 1 (answer)
        Config 3: 1 (answer)
        Config 4: 2 (1 query generation + 1 answer)
        Config 5: 1 (answer; CrossEncoder uses local model, 0 API calls)
    """

'\n    End-to-end RRF RAG pipeline orchestrator.\n\n    LLM call counts per query:\n        Config 1: 1 (answer)\n        Config 2: 1 (answer)\n        Config 3: 1 (answer)\n        Config 4: 2 (1 query generation + 1 answer)\n        Config 5: 1 (answer; CrossEncoder uses local model, 0 API calls)\n    '

In [47]:
config = RRFConfig()
logger.info("=== RRF (Reciprocal Rank Fusion) RAG Pipeline Starting ===")


2026-05-22 09:01:48 | INFO     | rrf_rag | === RRF (Reciprocal Rank Fusion) RAG Pipeline Starting ===


In [48]:
pdf_paths   = download_pdfs(config)


2026-05-22 09:01:48 | INFO     | rrf_rag | Cached: attention_is_all_you_need.pdf  (2163.3 KB)
2026-05-22 09:01:48 | INFO     | rrf_rag | Cached: bert_pretraining_transformers.pdf  (757.0 KB)
2026-05-22 09:01:48 | INFO     | rrf_rag | Cached: rag_knowledge_intensive_nlp.pdf  (864.6 KB)


In [49]:
chunks      = load_and_chunk_documents(pdf_paths, config)


2026-05-22 09:01:49 | INFO     | rrf_rag |   attention_is_all_you_need.pdf -> 15 pages -> 104 chunks
2026-05-22 09:01:52 | INFO     | rrf_rag |   bert_pretraining_transformers.pdf -> 16 pages -> 173 chunks
2026-05-22 09:01:52 | INFO     | rrf_rag |   rag_knowledge_intensive_nlp.pdf -> 19 pages -> 181 chunks
2026-05-22 09:01:52 | INFO     | rrf_rag | Total chunks: 458


In [50]:
embeddings  = build_bge_embeddings(config)


2026-05-22 09:01:52 | INFO     | rrf_rag | Loading BGE: BAAI/bge-large-en-v1.5
2026-05-22 09:01:52 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-05-22 09:01:53 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:01:53 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-22 09:01:53 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:01:53 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4342.48it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-22 09:01:56 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:01:56 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-22 09:01:57 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:01:57 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-22 09:01:57 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-22 09:01:57 | INFO     | httpx |

In [51]:
vectorstore = build_faiss_index(chunks, embeddings, config)


2026-05-22 09:01:58 | INFO     | rrf_rag | Loading FAISS from './faiss_rrf_index' ...
2026-05-22 09:01:58 | INFO     | rrf_rag |   Loaded. Vectors: 458


In [52]:
bm25_base   = build_bm25_index(chunks, config)


2026-05-22 09:01:58 | INFO     | rrf_rag | Loading BM25 from 'bm25_rrf_index\bm25plus.pkl' ...


In [53]:
llm         = build_ollama_llm(config)


In [54]:
demo_questions = [
    # Exact-keyword query -- BM25 should contribute strongly; k=20 adaptive
    "What BLEU score did the Transformer base model achieve on WMT 2014 English-to-German?",

    # Semantic query -- dense retriever leads; k=60 standard
    "How does multi-head attention allow the model to attend to information from different representation subspaces?",

    # Hybrid query -- both retrievers contribute; k=120 max consensus
    "What training procedure and optimizer are used in the Transformer model?",

    # Multi-hop: RAG-Fusion query variant diversity is most valuable here
    "How do BERT and the original Transformer differ in their use of attention for sequence understanding?",

    # Ambiguous query: precise reranking (CrossEncoder Config 5) most valuable
    "What is the role of the encoder in sequence-to-sequence tasks?",
]


In [55]:
# Run k sweep analysis on first query to illustrate k sensitivity
k_sweep_analysis(demo_questions[0], vectorstore, bm25_base, config=config)



RRF k SWEEP ANALYSIS
Query: What BLEU score did the Transformer base model achieve on WMT 2014 Eng
BM25 top-1: 6 Results
6.1 Machine Translation
On the WMT 2014 English-to...
Dense top-1: 6 Results
6.1 Machine Translation
On the WMT 2014 English-to...
     k  Rank1 doc (paper p#, rrf_score)                   
------------------------------------------------------------------------------
  k=   5:  [Attention Is All You p7]  score=0.333333
  k=  20:  [Attention Is All You p7]  score=0.095238
  k=  60:  [Attention Is All You p7]  score=0.032787
  k= 120:  [Attention Is All You p7]  score=0.016529
  k= 300:  [Attention Is All You p7]  score=0.006645



In [56]:
# Run all 5 configs on all demo questions
for question in demo_questions:
    run_all_configs(question, vectorstore, bm25_base, llm, config)

logger.info("=== RRF RAG Pipeline Demo Complete ===")



##############################################################################
QUERY: What BLEU score did the Transformer base model achieve on WMT 2014 English-to-German?
##############################################################################

RUNNING: Config1_Baseline
2026-05-22 09:02:15 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

TRACE: Config1_Baseline_SingleRetriever
Query: What BLEU score did the Transformer base model achieve on WMT 2014 English-to-Ge

  k constant used: 0
  Candidates pre-rerank: 0
  Final docs to LLM:     4

  Retriever hits:
    FAISS_BGE                               : 4 docs

  Final docs:
    [1] Attention Is All You Need p7  | rrf=n/a
    [2] Attention Is All You Need p7  | rrf=n/a
    [3] Attention Is All You Need p0  | rrf=n/a
    [4] Attention Is All You Need p7  | rrf=n/a

  Latency: retrieval=303ms  rerank=0ms  gen=14105ms  total=14409ms

  ANSWER:
  The provided documents do not contain sufficie

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4045.59it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-22 09:02:46 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:02:46 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/config.json "HTTP/1.1 200 OK"
2026-05-22 09:02:47 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:02:47 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-22 09:02:47 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-22 09:02:47 | INFO     | ht

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6953.12it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-22 09:03:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:03:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/config.json "HTTP/1.1 200 OK"
2026-05-22 09:03:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:03:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-22 09:03:41 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-22 09:03:41 | INFO     | ht

...

2026-05-22 09:04:32 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:04:32 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/config.json "HTTP/1.1 200 OK"
2026-05-22 09:04:33 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:04:33 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-22 09:04:33 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-22 09:04:33 | INFO     | ht

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6661.47it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-22 09:05:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:05:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/config.json "HTTP/1.1 200 OK"
2026-05-22 09:05:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:05:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-22 09:05:20 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-22 09:05:20 | INFO     | ht

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7277.95it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-22 09:06:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:06:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/config.json "HTTP/1.1 200 OK"
2026-05-22 09:06:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-22 09:06:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-22 09:06:44 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-22 09:06:44 | INFO     | ht